# Modélisation et Évaluation (Prédiction des Prix Immobiliers)

Ce notebook fait suite à l'analyse exploratoire et au nettoyage des données. L'objectif est d'entraîner plusieurs algorithmes de Machine Learning (régression linéaire et arbres), de comparer leurs performances, et d'optimiser le meilleur modèle pour un futur déploiement.

In [3]:
import os

import pandas as pd
import numpy as np

from boruta import BorutaPy

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from sklearn.ensemble import HistGradientBoostingRegressor

from sklearn.model_selection import GridSearchCV

import joblib
from sklearn.inspection import permutation_importance

### 1. Importation et Préparation Stratifiée des Données

**Choix techniques pour la préparation :**
- **Stratification (qcut) :** Pour garantir que le jeu d'entraînement et le jeu de test contiennent la même proportion de biens abordables et de biens luxueux, nous découpons la variable cible (`log_price`) en 5 tranches équilibrées lors de la séparation.
- **Target Encoding :** Le code postal (`zipcode`) est remplacé par le prix moyen du quartier, calculé *strictement* sur le jeu d'entraînement pour éviter la fuite de données

In [5]:
# 1. Chargement des données nettoyées
df = pd.read_csv('../data/king_county_cleaned.csv')

# 2. Suppression des colonnes non prédictives
cols_to_drop = ['Unnamed: 0', 'id', 'date', 'price', 'yr_built', 'yr_renovated']
df_model = df.drop(columns=cols_to_drop, errors='ignore')

# 3. Séparation des features (X) et de la cible (y)
X = df_model.drop(columns=['log_price'])
y = df_model['log_price']

# 4. Stratification : création de 5 tranches de prix équilibrées
strat_bins = pd.qcut(y, q=5, labels=False)

# 5. Séparation Train / Test stratifiée (80 % / 20 %)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=strat_bins
)

# 6. Target Encoding sur le 'zipcode'
zipcode_target_mean = y_train.groupby(X_train['zipcode']).mean()
X_train['zipcode_encoded'] = X_train['zipcode'].map(zipcode_target_mean)
X_test['zipcode_encoded'] = X_test['zipcode'].map(zipcode_target_mean)

# Remplacement des éventuelles valeurs inconnues dans le Test
X_test['zipcode_encoded'] = X_test['zipcode_encoded'].fillna(y_train.mean())

# Suppression de la colonne d'origine
X_train = X_train.drop(columns=['zipcode'])
X_test = X_test.drop(columns=['zipcode'])

print(f"X_train stratifié : {X_train.shape}")
print(f"X_test stratifié  : {X_test.shape}")

X_train stratifié : (17126, 20)
X_test stratifié  : (4282, 20)


In [6]:
# Boruta s'appuie sur un Random Forest pour juger les variables
# On limite la profondeur (max_depth=5) pour que le calcul soit rapide
rf_boruta = RandomForestRegressor(n_jobs=-1, max_depth=5, random_state=42)

# Initialisation de l'algorithme Boruta
# max_iter=50 limite le nombre de luttes contre les variables aléatoires
feat_selector = BorutaPy(
    rf_boruta, 
    n_estimators='auto', 
    verbose=2, 
    random_state=42, 
    max_iter=50
)

# Entraînement
print("Boruta")
feat_selector.fit(X_train.values, y_train.values)

# Extraction et affichage 
selected_features = X_train.columns[feat_selector.support_].tolist()
rejected_features = X_train.columns[~feat_selector.support_].tolist()

print(f"\n--- RÉSULTATS BORUTA ---")
print(f"Variables validées (pouvoir prédictif +) : {selected_features}")
print(f"Variables rejetées (bruit)  : {rejected_features}")

Lancement de Boruta (cela peut prendre 1 à 2 minutes)...
Iteration: 	1 / 50
Confirmed: 	0
Tentative: 	20
Rejected: 	0
Iteration: 	2 / 50
Confirmed: 	0
Tentative: 	20
Rejected: 	0
Iteration: 	3 / 50
Confirmed: 	0
Tentative: 	20
Rejected: 	0
Iteration: 	4 / 50
Confirmed: 	0
Tentative: 	20
Rejected: 	0
Iteration: 	5 / 50
Confirmed: 	0
Tentative: 	20
Rejected: 	0
Iteration: 	6 / 50
Confirmed: 	0
Tentative: 	20
Rejected: 	0
Iteration: 	7 / 50
Confirmed: 	0
Tentative: 	20
Rejected: 	0
Iteration: 	8 / 50
Confirmed: 	11
Tentative: 	4
Rejected: 	5
Iteration: 	9 / 50
Confirmed: 	11
Tentative: 	4
Rejected: 	5
Iteration: 	10 / 50
Confirmed: 	11
Tentative: 	4
Rejected: 	5
Iteration: 	11 / 50
Confirmed: 	11
Tentative: 	4
Rejected: 	5
Iteration: 	12 / 50
Confirmed: 	12
Tentative: 	3
Rejected: 	5
Iteration: 	13 / 50
Confirmed: 	12
Tentative: 	3
Rejected: 	5
Iteration: 	14 / 50
Confirmed: 	12
Tentative: 	3
Rejected: 	5
Iteration: 	15 / 50
Confirmed: 	12
Tentative: 	3
Rejected: 	5
Iteration: 	16 / 50
Co

**Analyse des résultats de Boruta et gestion de la colinéarité :**

L'exécution de l'algorithme Boruta est très instructive car elle vient confirmer mathématiquement l'analyse de colinéarité réalisée lors de l'exploration des données :

*   **Confirmation des variables maîtresses :** L'algorithme valide sans surprise le fort pouvoir prédictif de nos variables clés liées à l'espace (`sqft_living`, `sqft_lot`) et à la localisation géographique (`zipcode_encoded`, `lat`, `long`).
*   **Mise en évidence de la colinéarité (Variables rejetées) :** Boruta a classé des variables comme `bedrooms`, `bathrooms`, `floors` ou `sqft_basement` parmi les variables rejetées. Cela s'explique par leur **forte colinéarité** avec la surface habitable totale (`sqft_living`). Puisque le modèle possède déjà l'information de la surface globale, le nombre de chambres n'apporte mathématiquement pas d'information supplémentaire et fait doublon.
*   **Les limites de la sélection automatique :** L'algorithme a également rejeté des variables d'état comme `condition` et `age_effectif`. Pourtant, l'analyse finale de notre modèle champion (Gradient Boosting) démontrera que ces variables font partie du Top 10 des critères les plus détermte: 1]. Boruta, s'appuyant sur des arbres peu profonds, peine à capter ces subtilités.

**Conclusion sur la sélection :** 
Plutôt que de supprimer aveuglément les variables rejetées par Boruta, nous avons décidé de conserver celles qui ont un sens "métier" (comme l'âge effectif et la condition) et de laisser nos algorithmes de régression (notamment la régularisation de l'algorithme Ridge) gérer mathématiquement ces colinéarités résiduelles.

### 2. Modèle 1 : Régression Linéaire Régularisée (Ridge)

Nous débutons par un modèle linéaire pénalisé (Ridge).

**Explication des choix :**
- **RobustScaler :** Les algorithmes linéaires sont très sensibles à l'échelle. `RobustScaler` utilise la médiane et l'écart interquartile, offrant une mise à l'échelle robuste face aux propriétés exceptionnelles (asymétrie) qui subsistent sur le marché.
- **Validation Croisée In-Memory :** Notre jeu de données tient entièrement en mémoire vive (Full-Batch). `RidgeCV` découpe automatiquement les données d'entraînement en 5 plis croisés pour trouver la meilleure pénalité $\alpha$, sans nécessiter d'apprentissage par petits lots.
- **Métriques (Régression) :** Nous utilisons le **R²** (variance expliquée) et la **MAE** (erreur moyenne absolue reconvertie en dollars), qui sont les standards métiers.

In [7]:
# 1. Grille de pénalité alpha
alphas = np.logspace(-3, 3, 100)

# 2. Pipeline Ridge avec RobustScaler
ridge_pipeline = Pipeline([
    ('scaler', RobustScaler()),  
    ('model', RidgeCV(alphas=alphas, cv=5, scoring='neg_mean_squared_error'))
])

# 3. Entraînement sur le jeu Train
ridge_pipeline.fit(X_train, y_train)

best_alpha = ridge_pipeline.named_steps['model'].alpha_
print(f"Meilleur paramètre alpha retenu par RidgeCV : {best_alpha:.4f}\n")

# 4. Prédictions sur le Test et reconversion
y_pred_log = ridge_pipeline.predict(X_test)
y_test_real = np.exp(y_test)
y_pred_real = np.exp(y_pred_log)

# 5. Évaluation
r2_log = r2_score(y_test, y_pred_log)
mae_dollars = mean_absolute_error(y_test_real, y_pred_real)
rmse_dollars = np.sqrt(mean_squared_error(y_test_real, y_pred_real))

print("--- PERFORMANCE RIDGE (RobustScaler) ---")
print(f"Score R² : {r2_log:.4f}")
print(f"Erreur Moyenne Absolue (MAE) : {mae_dollars:,.2f} $")
print(f"Erreur Quadratique (RMSE)    : {rmse_dollars:,.2f} $")

Meilleur paramètre alpha retenu par RidgeCV : 1.8738

--- PERFORMANCE RIDGE (RobustScaler) ---
Score R² : 0.8607
Erreur Moyenne Absolue (MAE) : 83,428.44 $
Erreur Quadratique (RMSE)    : 163,671.49 $


#### Analyse des résultats du Modèle Ridge (Baseline Linéaire)

Ce premier modèle linéaire régularisé fixe un excellent point de repère (*baseline*) pour la suite du projet :

- **Pouvoir explicatif élevé ($R^2 \approx 86 \%$) :** Le modèle capte la très grande majorité de la variance des prix. Cela valide l'efficacité du prétraitement, en particulier la transformation logarithmique de la cible et le *Target Encoding* sur le code postal.
- **Paramètre $\alpha$ modéré ($1{,}87$) :** La validation croisée n'a pas eu besoin d'écraser fortement les coefficients, signe que notre sélection de variables a bien évité la redondance excessive (multicollinéarité).
- **Une marge d'erreur perfectible (MAE $\approx 83\ 400\ \$$) :** L'écart moyen reste élevé pour fixer un prix de vente définitif. L'important écart entre la MAE et la **RMSE ($163\ 671\ \$$)** souligne la limite d'un modèle strictement additif : il sous-estime lourdement les biens d'exception (haut de gamme) où les critères comme la vue, la surface et la localisation agissent ensemble de façon exponentielle.

**Conclusion :** Pour réduire ces grosses erreurs et descendre sous la barre des 80 000 $ de marge moyenne, nous devons basculer sur des algorithmes capables d'apprendre des relations **non linéaires**, comme les arbres de décision.

### 3. Modèle 2 : Arbres de Décision Ensemblistes (Random Forest)

Contrairement au modèle linéaire, les Random Forests n'ont pas besoin de mise à l'échelle (`RobustScaler`) et gèrent nativement les relations non linéaires (ex: l'interaction entre une belle vue et une grande surface).

In [8]:
# 1. Instanciation du Random Forest (n_jobs=-1 utilise tous les processeurs)
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

# 2. Entraînement direct sur les données brutes
rf_model.fit(X_train, y_train)

# 3. Prédictions et reconversion
y_pred_log_rf = rf_model.predict(X_test)
y_pred_real_rf = np.exp(y_pred_log_rf)

# 4. Évaluation
r2_log_rf = r2_score(y_test, y_pred_log_rf)
mae_dollars_rf = mean_absolute_error(y_test_real, y_pred_real_rf)
rmse_dollars_rf = np.sqrt(mean_squared_error(y_test_real, y_pred_real_rf))

print("--- PERFORMANCE RANDOM FOREST ---")
print(f"Score R² : {r2_log_rf:.4f}")
print(f"Erreur Moyenne Absolue (MAE) : {mae_dollars_rf:,.2f} $")
print(f"Erreur Quadratique (RMSE)    : {rmse_dollars_rf:,.2f} $")

--- PERFORMANCE RANDOM FOREST ---
Score R² : 0.8935
Erreur Moyenne Absolue (MAE) : 68,558.58 $
Erreur Quadratique (RMSE)    : 126,864.85 $


#### Analyse des résultats du Random Forest

Le passage à un modèle ensembliste non linéaire apporte un gain prédictif net :
- Le $R^2$ atteint près de 90 % ($0{,}8935$).
- L'erreur moyenne absolue (MAE) diminue de près de 15 000 $ pour s'établir à environ 68 500 $.
- La chute de la RMSE (-36 800 $) confirme que les arbres corrigent les sous-estimations systématiques du modèle linéaire sur les biens d'exception en combinant naturellement l'emplacement, la vue et le standing.

### 4. Modèle 3 : Gradient Boosting (HistGradientBoosting)

Pour notre troisième modèle, nous utilisons un algorithme de *Boosting*. Contrairement au Random Forest (*Bagging*) qui construit des arbres indépendants en parallèle, le *Boosting* construit les arbres de manière séquentielle : chaque nouvel arbre tente de corriger les erreurs résiduelles de l'arbre précédent. 

L'implémentation `HistGradientBoostingRegressor` de Scikit-Learn est optimisée pour les grands jeux de données en regroupant les valeurs continues en histogrammes, ce qui accélère drastiquement l'entraînement sans perte de précision.

In [9]:
# 1. Instanciation du modèle
gb_model = HistGradientBoostingRegressor(
    max_iter=200,      # Nombre d'arbres
    learning_rate=0.1, # Vitesse d'apprentissage
    max_depth=7,       # Profondeur des arbres
    random_state=42
)

# 2. Entraînement sur les données brutes (pas besoin de RobustScaler)
gb_model.fit(X_train, y_train)

# 3. Prédictions
y_pred_log_gb = gb_model.predict(X_test)
y_pred_real_gb = np.exp(y_pred_log_gb)

# 4. Évaluation
r2_log_gb = r2_score(y_test, y_pred_log_gb)
mae_dollars_gb = mean_absolute_error(y_test_real, y_pred_real_gb)
rmse_dollars_gb = np.sqrt(mean_squared_error(y_test_real, y_pred_real_gb))

print("--- PERFORMANCE GRADIENT BOOSTING ---")
print(f"Score R² : {r2_log_gb:.4f}")
print(f"Erreur Moyenne Absolue (MAE) : {mae_dollars_gb:,.2f} $")
print(f"Erreur Quadratique (RMSE)    : {rmse_dollars_gb:,.2f} $")

--- PERFORMANCE GRADIENT BOOSTING ---
Score R² : 0.9071
Erreur Moyenne Absolue (MAE) : 64,382.76 $
Erreur Quadratique (RMSE)    : 123,199.74 $


#### Analyse des résultats du Gradient Boosting

Le passage au *Gradient Boosting* (apprentissage séquentiel par histogrammes) marque une nouvelle évolution majeure dans nos performances :

- **Franchissement du cap des 90 % ($R^2 = 0{,}9071$) :** Ce modèle explique désormais plus de 90 % de la variance du prix. En corrigeant itérativement les erreurs des arbres précédents, il capte beaucoup plus finement les spécificités locales que la forêt aléatoire.
- **Réduction continue de l'erreur (MAE $\approx 64\ 380\ \$$) :** L'écart moyen se resserre encore, gagnant plus de $4\ 100\ \$$ de précision par rapport au Random Forest, et près de $19\ 000\ \$$ face au modèle linéaire.
- **Meilleure gestion des prix extrêmes (RMSE $\approx 123\ 200\ \$$) :** La baisse de la RMSE confirme que la modélisation séquentielle parvient à mieux estimer les biens haut de gamme, qui posaient le plus de difficultés aux algorithmes précédents.

**Conclusion :** Le Gradient Boosting est clairement supérieur sur ce jeu de données. Il est logiquement retenu comme algorithme final. La prochaine et dernière étape consiste à ajuster ses hyperparamètres (*Grid Search*) pour tenter d'extraire les derniers pourcentages de performance avant son déploiement en production.

### 5. Optimisation du Modèle Final (GridSearchCV)

Afin d'extraire la performance maximale de notre algorithme retenu (`HistGradientBoostingRegressor`) avant son déploiement, nous procédons à une recherche exhaustive des hyperparamètres optimaux via `GridSearchCV`.

**Paramètres explorés (24 combinaisons testées en validation croisée sur 3 plis) :**
- **`learning_rate` (0.05, 0.1) :** Contrôle la contribution de chaque nouvel arbre. Un taux plus faible nécessite généralement plus d'arbres, mais permet d'éviter les optimums locaux.
- **`max_iter` (150, 250) :** Définit le nombre total d'arbres construits séquentiellement.
- **`max_depth` (6, 10, None) :** Limite la profondeur de chaque arbre pour contrôler la complexité des règles créées. `None` laisse les arbres croître sans limite de niveau.
- **`min_samples_leaf` (20, 50) :** Impose un nombre minimum de biens par feuille (nœud final), servant de régulateur puissant pour éviter le surapprentissage (*overfitting*) sur des micro-quartiers ou des cas exceptionnels.

In [10]:
# 1. Grille ciblée d'hyperparamètres (24 combinaisons testées)
param_grid = {
    'learning_rate': [0.05, 0.1],
    'max_iter': [150, 250],
    'max_depth': [6, 10, None],
    'min_samples_leaf': [20, 50]
}

# 2. Instanciation du GridSearchCV (cv=3 pour un calcul rapide)
grid_gb = GridSearchCV(
    estimator=HistGradientBoostingRegressor(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

# 3. Entraînement et recherche du meilleur profil
grid_gb.fit(X_train, y_train)

print(f"Meilleurs hyperparamètres : {grid_gb.best_params_}\n")

# 4. Prédictions avec le modèle optimisé sur le jeu de test
best_gb = grid_gb.best_estimator_
y_pred_log_opt = best_gb.predict(X_test)
y_pred_real_opt = np.exp(y_pred_log_opt)

# 5. Évaluation finale
r2_opt = r2_score(y_test, y_pred_log_opt)
mae_opt = mean_absolute_error(y_test_real, y_pred_real_opt)
rmse_opt = np.sqrt(mean_squared_error(y_test_real, y_pred_real_opt))

print("--- PERFORMANCE GRADIENT BOOSTING OPTIMISÉ ---")
print(f"Score R² : {r2_opt:.4f}")
print(f"Erreur Moyenne Absolue (MAE) : {mae_opt:,.2f} $")
print(f"Erreur Quadratique (RMSE)    : {rmse_opt:,.2f} $")

Meilleurs hyperparamètres : {'learning_rate': 0.05, 'max_depth': None, 'max_iter': 250, 'min_samples_leaf': 50}

--- PERFORMANCE GRADIENT BOOSTING OPTIMISÉ ---
Score R² : 0.9085
Erreur Moyenne Absolue (MAE) : 62,971.58 $
Erreur Quadratique (RMSE)    : 119,208.69 $


#### Analyse des Hyperparamètres et du Modèle Final Optimisé

L'optimisation via `GridSearchCV` permet de franchir un ultime palier de performance :

- **Profil optimal retenu :** Le modèle privilégie un taux d'apprentissage plus modéré (`learning_rate = 0.05`) combiné à un nombre d'itérations plus élevé (`max_iter = 250`), ce qui stabilise la descente de gradient. L'absence de restriction sur la profondeur (`max_depth = None`) est contrebalancée par une forte contrainte sur les feuilles (`min_samples_leaf = 50`), empêchant l'algorithme d'apprendre par cœur les biens atypiques.
- **Précision maximale ($R^2 = 0{,}9085$) :** Plus de 90,8 % de la variance des prix est expliquée, établissant le record de performance sur l'ensemble de l'étude.
- **Gain métier sur l'erreur moyenne (MAE $\approx 62\ 970\ \$$) :** L'optimisation réduit encore l'écart moyen d'environ $1\ 400\ \$$ par rapport au Gradient Boosting initial, et de plus de $20\ 450\ \$$ face au modèle linéaire de référence (*baseline*). L'erreur relative moyenne descend autour de 14 %.
- **Amélioration sur les cas extrêmes (RMSE $< 120\ 000\ \$$) :** La chute sous la barre des $120\ 000\ \$$ confirme une meilleure tolérance aux résidences de prestige.

### 6. Interprétabilité (Feature Importance) et Sauvegarde du Modèle

Pour valider la logique métier du modèle et préparer son intégration dans l'application web :
1. Nous calculons l'importance des variables par permutation (*Permutation Importance*) sur le jeu de test. Cette approche mesure la perte de performance lorsque les valeurs d'une variable sont mélangées aléatoirement.
2. Nous exportons le modèle final optimisé ainsi que la table de correspondance du *Target Encoding* des codes postaux avec `joblib`.

In [11]:
# 1. Calcul de l'importance par permutation sur le jeu de test
perm = permutation_importance(best_gb, X_test, y_test, n_repeats=10, random_state=42)

# 2. Tableau des 10 variables les plus déterminantes
indices = perm.importances_mean.argsort()[::-1]
df_importance = pd.DataFrame({
    'Variable': X_test.columns[indices],
    'Importance': perm.importances_mean[indices]
})

print("--- TOP 10 DES VARIABLES LES PLUS IMPORTANTES ---")
print(df_importance.head(10).to_string(index=False))

--- TOP 10 DES VARIABLES LES PLUS IMPORTANTES ---
       Variable  Importance
zipcode_encoded    0.231167
    sqft_living    0.195638
            lat    0.158926
          grade    0.094196
           long    0.052497
       sqft_lot    0.033205
           view    0.017112
   age_effectif    0.013486
  sqft_living15    0.011398
      condition    0.009100


#### Analyse de l'Importance des Variables (Cohérence Métier)

L'analyse de l'importance par permutation sur le jeu de test met en lumière la hiérarchie logique des critères de valorisation :

1. **La localisation ultra-dominante (~44 % de l'impact combiné) :**
   - Le code postal encodé (`zipcode_encoded` : 0,231) et les coordonnées géographiques fines (`lat` : 0,159 et `long` : 0,052) constituent le premier pilier du prix. Le modèle capte non seulement la réputation globale du quartier, mais aussi des micro-zones à forte valeur ajoutée (proximité du lac Washington, vue, centres économiques).
2. **La surface habitable (`sqft_living` : 0,196) :**
   - Sans surprise, la superficie intérieure reste le principal facteur intrinsèque du bien.
3. **La qualité de construction (`grade` : 0,094) :**
   - La qualité des matériaux et du design pèse bien plus lourd que le simple état d'entretien (`condition` : 0,009). Un bien de conception haut de gamme conserve une prime forte, même s'il nécessite quelques rafraîchissements.
4. **Les facteurs d'agrément et d'époque :**
   - La superficie du terrain (`sqft_lot`), la présence d'une vue (`view`) et l'âge effectif du bien (`age_effectif` : année de construction ou dernière rénovation) complètent l'ajustement fin**Conclusion**: Notebook :** Le modèle s'appuie sur des critères rationnels, sans biest terminée.

In [12]:
# 3. Création du dossier et sauvegarde des artefacts pour Flask
os.makedirs('../model', exist_ok=True)

joblib.dump(best_gb, '../model/gradient_boosting_model.joblib')
joblib.dump(zipcode_target_mean.to_dict(), '../model/zipcode_encoding.joblib')

print("\nFichiers sauvegardés avec succès dans le dossier 'model' :")
print("- gradient_boosting_model.joblib")
print("- zipcode_encoding.joblib")


Fichiers sauvegardés avec succès dans le dossier 'model' :
- gradient_boosting_model.joblib
- zipcode_encoding.joblib
